# Sucking Problem

This is a testcase for evaporation.  
Some liquid evaporated at a planar fluid vapor interface.  
The evaporation is driven by an initially superheated liquid.  
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* If we initialize with zero velocity fields the interface movement is off in the first timestep. There are two workarounds: initialize with the analytical velocity (done now) or use a very small first timestep.
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "SuckingProblem";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

We make use of the following connections ($A$ - Liquid, $B$ - Vapor),
as the problem is quasi-1D, only the x-directional velocity components are considerd :  
Known values from vapor side:    
* $T_B(x,t) = T_{sat}$
* $u_B(x,t) = 0$
Interface velocity :  
* $x_I(t) = 2 * \xi * \sqrt{\alpha_B * t}$
* $\dot{x}_I(t) = \xi * \sqrt{\frac{\alpha_B}{t}}$
* $\xi = 0.7677$
* $\alpha_{A/B} = \frac{k_{A/B}}{\rho_{A/B}c_{A/B}}$  
Fluxes at the interface : 
* $\dot{m} = -\rho_B(u_B - \dot{x}) = \rho_B \dot{x}$
* $h_V\dot{m} = k_A T_{x,A}$  
Therefore :  
* $T_{x,A} = \frac{\rho_B h_V}{k_A} \xi \sqrt{\alpha_B t}$  
this temperature gradient at the interface is used for extrapolating the temperature profile in phase $A$ for setting the initial conditions.

Rescale temperature, for some reason better for the solver:  
$\Theta = \frac{T - T_{s}}{T_{s} - T_{Wall}}$  
$\Delta T = {T_{s} - T_{Wall}}$  

$ \tilde{c} = \Delta T * c$  
$ \tilde{k} = \Delta T * k$  

In [ ]:
static class Constants{
    // careful order of declaration matters!!!
    public static double L = 0.01;
    public static double Xi = 0.7677; // see Matlab Sheet SuckingProblem.m, note the different formulas for interface position in SuckingProblem.m and the reference work. In the following we use the constants stated in the reference.

    // material parameters
    public static double rho_A = 958.4;
    public static double rho_B = 0.597;

    public static double mu_A = 2.8e-4;
    public static double mu_B = 1.26e-5;

    public static double Sigma = 0.059 / 1000.0; // very small surface tension so that capillary timestep is bigger...

    public static double c_A = 4216.0 * 5.0;
    public static double c_B = 2030.0 * 5.0;

    public static double k_A = 0.679 * 5.0;
    public static double k_B = 0.025 * 5.0;

    public static double hVap  = 2.258e6;
    public static double T_sat = 0.0;//373.15;
    public static double T_wall = T_sat + 1.0;//T_sat + 5.0;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);    
    public static double C = k_A / (rho_B*hVap);
    public static double eps = 1.0-rho_B/rho_A;

    public static double x0 = 0.00221;
    public static double t0 = Math.Pow(0.5 * x0 / Xi, 2) / alpha_B; // start time, to avoid singular massflux at t=0
    public static double tE = 0.6 - t0; // Endtime
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(0, Constants.L, res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, h, 2); // quasi 1D always use one cell
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "pressure_outlet_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if(Math.Abs(X[0]) < 1e-6)
                return 2;
            if(Math.Abs(X[0] -  Constants.L) < 1e-6)
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_B + " / Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine("         return InterfacePos()(X,t) - X[0];");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + "; // always saturation temp.");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureABoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double TemperatureBBoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }
   
    static public Formula Get_VA() {
        return new Formula("BoundaryAndInitialValues.VelocityA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureBBoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureABoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

In [ ]:
// load reference solution data, see Matlab Sheet SuckingProblem.m, to initialize the temperature profile. This is also a similarity solution, used as reference later on
string path = @".\SuckingProblemRef.csv";
string[] lines = File.ReadAllLines(path);
double[] eta = new double[lines.Length];
double[][] valueDat = new double[2][];
for(int j = 0; j < 2; j++)
    valueDat[j] = new double[lines.Length];

for (int i = 0; i < lines.Length; i++) {
    eta[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
    for (int j = 0; j < 2; j++) {
        valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
        if(j == 0) valueDat[j][i] = (valueDat[j][i] - 373.15) / 5.0;
    }
} 

In [ ]:
// use loaded data, to construct a spline as the initial condition
double x0 = Constants.x0;
double[] x = new double[lines.Length+2];
double[] y = new double[lines.Length+2];
x[0] = 0.0;
y[0] = Constants.T_sat-x0 * Constants.Xi / Constants.C * Math.Sqrt(Constants.alpha_B / Constants.t0) ; // linear extrapolation node, using slope of temperature at interface
for(int i = 1; i < lines.Length + 1; i++){
    y[i] = valueDat[0][i-1];
    double xi = eta[i-1]* Math.Sqrt(2 * Constants.alpha_A * Constants.t0);
    x[i] = x0 + xi;
}
x[lines.Length+1] = Constants.L;
y[lines.Length+1] = Constants.T_wall;
Spline1D TempAInitial = new Spline1D(x,y,0);

In [ ]:
Gnuplot plt = new Gnuplot();
plt.PlotFunction(delegate (MultidimensionalArray X, MultidimensionalArray Y) {TempAInitial.EvaluateV(X,0.0,Y);}, BoSSS.Foundation.Grid.Classic.Grid1D.LineGrid(GenericBlas.Linspace(Constants.x0,0.008,1000)).GridData);
plt.PlotNow()

## Begin Postprocessing

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions

Lets sort the sessions by degree

In [ ]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

Load Log Data

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

### Interface Position

Evaluate the analyical solution for the interface position

In [ ]:
public static void subsample(Plot2Ddata plt, int n){
    foreach(var group in plt.dataGroups){
        var X = group.Abscissas;
        var Y = group.Values;
        int N = X.Count();
        List<double> newX = new List<double>();
        List<double> newY = new List<double>();
        
        newX.Add(X[0]);
        newY.Add(Y[0]);
        for(int i = 1; i < N - 1; i++){
            if((i % n) == 0){
                newX.Add(X[i]);
                newY.Add(Y[i]);
            }
        }
        newX.Add(X[N-1]);
        newY.Add(Y[N-1]);
        
        group.Abscissas = newX.ToArray();
        group.Values = newY.ToArray();
    }
}

In [ ]:
data.ForEach(dat => dat.Value.ForEach(pp => subsample(pp, 50)));

In [ ]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE + Constants.t0, 1000);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[2], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i] - Constants.t0);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "analytical Interface Position");
RefPos.dataGroups.First().Format = new PlotFormat("r-");

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
     foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(1).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    };
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var p1 = plot.PlotNow(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE ;
    plot.XrangeMax = Constants.tE+ Constants.t0;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    // display(pp);
}

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
data.ForEach(dat => dat.Value.ForEach(pp => subsample(pp, 50)));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
     foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(1).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    };
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var p1 = plot.PlotNow(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE ;
    plot.XrangeMax = Constants.tE+ Constants.t0;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Repeat the same for positional errors

In [ ]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

### Spatial Convergence

Evaluated at last timestep  
Plot Positional error over hmin

In [ ]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    double t_End = data[k].ElementAt(0).dataGroups.First().Abscissas.Last();
    double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);

    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["Grid:hMin"]);
        var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "P = " + k));    
}

p.ModFormat();
//p.AddDataGroup(" O(1) ", new[] {1e-4, 1e-2}, new[] {1e-4, 1e-2}, "r-");
p.AddDataGroup(" O(2) ", new[] {1e-4, 1e-2}, new[] {1e-5, 1e-1}, "r-");
//p.AddDataGroup(" O(3) ", new[] {1e-4, 1e-3}, new[] {1e-5, 1e-2}, "r--");
p.LogX = true;
p.LogY = true;
var gp = p.ToGnuplot();
// gp.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_SpatialConvergence"); // change a few settings by hand, see readme!

HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        p.PlotNow().ToString() +
    "   </div>" +
    "</div>");
display(pp);

### Growthrate


In [ ]:
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();

Lets sort the sessions by degree

In [ ]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

Load Log Data

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

Calculate the $L1$ relative error in the computed growthconstant:  
$E_1 = \frac{1}{t_{max}-t_0}\sum_i|\frac{x_I}{x_{I,ex}}-1|\Delta t_i$  
or as we have equally spaced timesteps:  
$E_1 = \frac{1}{N}\sum_i|\frac{x_I}{x_{I,ex}}-1|$

In [ ]:
static Formula _AnalyticalInterfacePos = BoundaryAndInitialValueFactory.Get_Phi();
static Func<double, double> AnalyticalInterfacePos = t => {
    return _AnalyticalInterfacePos.Evaluate(new double[2], t);
};

public static double RelErrorGrowthConstant(Plot2Ddata.XYvalues InterfacePos){
    double err = 0.0;
    
    int N = InterfacePos.Abscissas.Length;
    for(int i = 0; i < N; i++){
        err += Math.Abs(InterfacePos.Values[i]/AnalyticalInterfacePos(InterfacePos.Abscissas[i]) - 1.0);
    }
    err = err/N;

    return err;
}

public static (string, double[], double[]) ReadReferenceCSV(string file){
    List<double> X = new List<double>();
    List<double> Y = new List<double>();
    string name;
    using(var reader = new StreamReader(file))
    {
        var line = reader.ReadLine();
        var values = line.Split(';');    
        name = "Bureš, Sato : " + values[1] + " problem";

        while (!reader.EndOfStream)
        {
            line = reader.ReadLine();
            values = line.Split(';');

            X.Add(Convert.ToDouble(values[0].Replace(',','.')));
            Y.Add(Convert.ToDouble(values[1].Replace(',','.')));
        }
    }
    return (name, X.ToArray(), Y.ToArray());
}

Finest simulation k=1,
final absolute error and relative mean error

In [ ]:
var dd = data[1].ElementAt(0).dataGroups.First();
Math.Abs(AnalyticalInterfacePos(dd.Abscissas.Second()) - dd.Values.Second()).Display();
RelErrorGrowthConstant(dd).Display();

Finest simulation k=3,
final absolute error and relative mean error

In [ ]:
var dd = data[3].ElementAt(0).dataGroups.First();
Math.Abs(AnalyticalInterfacePos(dd.Abscissas.Second()) - dd.Values.Second()).Display();
RelErrorGrowthConstant(dd).Display();

In [ ]:
data[1].ElementAt(0).dataGroups.First().Values.First().Display();
AnalyticalInterfacePos(data[1].ElementAt(0).dataGroups.First().Abscissas.First()).Display();

In [ ]:
Plot2Ddata GrowthConstantErrorRel = new Plot2Ddata();

foreach(int p in data.Keys.OrderBy(x => x)){
    if(p == 5) continue; // skip p=4
    var dat = data[p];

    double[] grid = new double[dat.ElementAt(0).dataGroups.Length];
    double[] error = new double[dat.ElementAt(0).dataGroups.Length];

    for(int i = 0; i < dat.ElementAt(0).dataGroups.Length; i++){
        var dataGroup = dat.ElementAt(0).dataGroups[i];
        grid[i] = 1.0/Convert.ToDouble(dataGroup.Name.Split('-')[1].Remove(0,1)) * 8; // normalize with base grid
        error[i] = RelErrorGrowthConstant(dataGroup) * 100.0; // in %

    }

    Plot2Ddata GrowthConstantErrorRel_p = new Plot2Ddata(grid, error, "k=" + p);
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_p);

}
{
    var Reference = ReadReferenceCSV("BuresSato2021_SuckingProblem_GrowthConstant.csv");
    Plot2Ddata GrowthConstantErrorRel_ref = new Plot2Ddata(Reference.Item2, Reference.Item3, Reference.Item1);
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_ref);
}

var plot = GrowthConstantErrorRel;
plot.LegendAlignment = new string[] {"i", "t", "l"}; 
plot.Ylabel = "Relative error [%]";
plot.Xlabel = "1/Grid level [-]";
plot.LabelFont = 16;
plot.LabelTitleFont = 16;
plot.Title = "Relative error in growth constant";
plot.TitleFont = 24;

plot.LogX = true;
plot.XrangeMin = 0.02;
plot.XrangeMax = 2;
plot.LogX2 = true;
plot.X2rangeMin = 0.02;
plot.X2rangeMax = 2;

plot.LogY = true;
plot.YrangeMin = 0.01;
plot.YrangeMax = 30.0;

plot.ModFormat();
var gp = plot.ToGnuplot();
var p2 = gp.PlotSVG(); 

HtmlString pp = new HtmlString(
    "<div>" +
    "   <div>" +
        p2.ToString() +
    "   </div style='float:right;'>" +
    "</div>");
display(pp);

In [ ]:
KeyValuePair<int, double>[] refslopes = new KeyValuePair<int, double>[] {new KeyValuePair<int, double>(1, 2), new KeyValuePair<int, double>(2, 1.4), new KeyValuePair<int, double>(3, 1.6), new KeyValuePair<int, double>(4, 1.4)};
var slopes = plot.Regression().Take(4).ToDictionary(x=> Convert.ToInt32(x.Key.Split("=").ElementAt(1)),x=>x.Value);
slopes.Display();

Comparison of DOFs:  
BoSSS on finest grid (first timestep)

In [ ]:
groupedSessions[3]

In [ ]:
var s = groupedSessions[3].First();
(s.GetDOF("VelocityX")+s.GetDOF("VelocityY")+s.GetDOF("Pressure")+s.GetDOF("Temperature")).Display()

Bures,Sato 2021 (estimated)

In [ ]:
Math.Pow(400, 1) * Math.Pow(8, 1) * 4

### Temporal Convergence

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "T");
//sessions = sessions.Where(s => s.Name.Contains("Splitting")).ToList();
//sessions = sessions.Where(s => s.Name.Contains("SDIRK") || s.Name.Contains("Euler") ).ToList();
sessions = sessions.Where(s => s.Name.Contains("LieSplitting")).ToList();
sessions = sessions.Where(s => s.Name.Contains("ImplicitEuler") && !s.Name.Contains("RK")).ToList();
sessions = sessions.OrderBy(s => s.KeysAndQueries["dtFixed"]).ToList();
sessions.ForEach(s => display(s))

In [ ]:
//Dictionary<int, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["Timestepper_LevelSetHandling"])).ToDictionary(g => g.Key, g => g.ToList());
Dictionary<Tuple<int, string>, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => new Tuple<int, string>(Convert.ToInt32(s.KeysAndQueries["Timestepper_LevelSetHandling"]), Convert.ToString((BoSSS.Solution.XdgTimestepping.TimeSteppingScheme)Convert.ToInt32(s.KeysAndQueries["TimeSteppingScheme"])))).ToDictionary(g => g.Key, g => g.ToList());
groupedSessions.Keys

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

In [ ]:
data.First().Value.ElementAt(0).dataGroups[0].Abscissas.Last()

In [ ]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    double t_End = data[k].ElementAt(0).dataGroups[i].Abscissas.Last();
    double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);
    
    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["dtFixed"]);
        var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "Splitting Order = " + k.Item1 + " Scheme : " + k.Item2));    
}

p.LegendAlignment = new[] {"o","c","r"};
p.ModFormat();
p.AddDataGroup(" O(1) ", new[] {1e-5, 1e-3}, new[] {1e-8, 1e-6}, "r-");
//p.AddDataGroup(" O(2) ", new[] {2e-5, 1e-4}, new[] {1e-8, 2.5e-7}, "r-");
var gp1 = p.ToGnuplot();
// gp1.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_TemporalConvergence"); // change a few settings by hand, see readme!

p.LogX = true;
p.LogY = true;
p.XrangeMin = 1 * 1e-5;
p.XrangeMax = 2 * 1e-4;
Gnuplot gp = new Gnuplot();
p.ToGnuplot(gp);
HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        gp.PlotSVG(xRes: 1600).ToString() +
    "   </div>" +
    "</div>");
display(pp);

In [ ]:
sessions.ForEach(s => display(s.KeysAndQueries["dtFixed"]));

In [ ]:
foreach(var p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

### L2 Convergence

We trick a little and use the exact solution to compare against not as a function of time, but of interface position! 

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions

In [ ]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

In [ ]:
// var sess = sessions.Pick(49);
// sess
//sess.Timesteps.Last().Export().WithShadowFields().To(Path.GetFullPath("./Plot/"+sess.Name)).WithSupersampling(2).Do()

#### Utility Functions

In [ ]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [ ]:
// Code functions of V_x, T, p as Functions of f(x, t(xI)) by inversing the relationship xI(t)
public static double InterfaceTime(double xI){
    return Math.Pow(xI / (2 * Constants.Xi),2) /  Constants.alpha_B - Constants.t0;
}

public static double InterfaceVel(double[] X, double t){
    return Constants.Xi * Constants.alpha_B / Math.Sqrt(Constants.alpha_B * (t+Constants.t0));
}

public static Func<double[], double>[] GetV(double xI){
    double tI = InterfaceTime(xI);
    return new Func<double[], double>[] { (X) => Constants.eps * InterfaceVel(X,tI), (X) => 0.0};
}

public static Func<double[], double>[] GetT(double xI){
    string path = @".\SuckingProblemRef.csv";
    string[] lines = File.ReadAllLines(path);
    double[] eta = new double[lines.Length];
    double[][] valueDat = new double[2][];
    for(int j = 0; j < 2; j++)
        valueDat[j] = new double[lines.Length];

    for (int i = 0; i < lines.Length; i++) {
        eta[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
        for (int j = 0; j < 2; j++) {
            valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            if(j == 0) valueDat[j][i] = (valueDat[j][i] - 373.15) / 5.0;
        }
    } 
    
    double _xI = xI;
    double _tI = InterfaceTime(xI);

    double[] x = new double[lines.Length+2];
    double[] y = new double[lines.Length+2];
    x[0] = 0.0;
    y[0] = Constants.T_sat-_xI * Constants.Xi / Constants.C * Math.Sqrt(Constants.alpha_B / (Constants.t0 + _tI)) ; // linear extrapolation node, using slope of temperature at interface
    for(int i = 1; i < lines.Length + 1; i++){
        y[i] = valueDat[0][i-1];
        double xi = eta[i-1]* Math.Sqrt(2 * Constants.alpha_A * (Constants.t0 + _tI));
        x[i] = _xI + xi;
    }
    x[lines.Length+1] = Constants.L;
    y[lines.Length+1] = Constants.T_wall;
    Spline1D TempAInitial = new Spline1D(x,y,0);

    Func<double[], double> TA = (X) => {        
        return TempAInitial.Evaluate(X,0.0);
    };

    return new Func<double[], double>[] { TA, (X) => 0.0};
}

public static double GetXI(ITimestepInfo ts){
    XDGField field = (XDGField)ts.Fields.Where(f => f is XDGField).First();
    var LsTrk = field.Basis.Tracker;
    var helper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS, 0).XQuadSchemeHelper;
    var rule = helper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask()).Compile(LsTrk.GridDat, 0);

    double x = 0.0;
    foreach(var part in rule){
        foreach(int i in part.Chunk.Elements){
            MultidimensionalArray glob = MultidimensionalArray.Create(part.Rule.Nodes.Lengths);
            LsTrk.GridDat.TransformLocal2Global(part.Rule.Nodes, glob, i);
            x = Math.Max(x, glob.ExtractSubArrayShallow(-1, 0).Max());
        }
    }

    //Console.WriteLine("ts : {0}, x : {1}", ts.Grid.NumberOfCells, x);
    return x;
}

In [ ]:
public static double ComputeErrorTemperature(ITimestepInfo ts, int mode = 0){    
    double xI = GetXI(ts);
    var exact = GetT(xI);
    var field = ts.Fields.Single(f => f.Identification == "Temperature").As<XDGField>();
    double err = ComputeErrors(field, exact, mode);
    return err;
}

public static double ComputeErrorVelocity(ITimestepInfo ts, int mode = 0){
    double xI = GetXI(ts);
    var exact = GetV(xI);
    //Console.WriteLine("Expected velocity {0}", exact[0](new[] {0.0, 0.0}));
    var field = ts.Fields.Single(f => f.Identification == "VelocityX").As<XDGField>();
    double err = ComputeErrors(field, exact, mode);
    //Console.WriteLine("Velocity error : {0}", err);
    return err;
}

public static double ComputeErrors(XDGField field, Func<double[], double>[] ExactSolution, int mode = 0){
    int order = 12;    
    var LsTrk = field.Basis.Tracker;
    int N = field.GridDat.iGeomCells.NoOfLocalUpdatedCells;
    double B = Constants.L / N;
    double b = Math.Sqrt(B);
    //Console.WriteLine(LsTrk.CutCellQuadratureType);
    var schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS, order).XQuadSchemeHelper;
    XDGField Solution = new XDGField(new XDGBasis(LsTrk, 6));
    Solution.GetSpeciesShadowField("A").ProjectField(ExactSolution[0]);
    Solution.GetSpeciesShadowField("B").ProjectField(ExactSolution[1]);

    var schemeA = schemeHelper.GetVolumeQuadScheme(LsTrk.SpeciesIdS[0]);

    double errA = 0.0;
    double areaA = 0.0;
    CellQuadrature.GetQuadrature(new[] { 2 }, LsTrk.GridDat, schemeA.Compile(LsTrk.GridDat, order), 
    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
        Solution.GetSpeciesShadowField("A").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), 1.0 );
        field.GetSpeciesShadowField("A").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), -1.0 );
        EvalResult.ExtractSubArrayShallow(-1, -1, 0).ApplyAll(x => x*x);
        EvalResult.ExtractSubArrayShallow(-1, -1, 1).SetAll(1.0);
    },
    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        for (int i = 0; i < Length; i++){
            errA += ResultsOfIntegration[i, 0];
            areaA += ResultsOfIntegration[i, 1];
        }
    }).Execute();
    errA = Math.Sqrt(errA);

    var schemeB = schemeHelper.GetVolumeQuadScheme(LsTrk.SpeciesIdS[1]);

    double errB = 0.0;
    double areaB = 0.0;
    CellQuadrature.GetQuadrature(new[] { 2 }, LsTrk.GridDat, schemeB.Compile(LsTrk.GridDat, order), 
    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
        Solution.GetSpeciesShadowField("B").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), 1.0 );
        field.GetSpeciesShadowField("B").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), -1.0 );
        EvalResult.ExtractSubArrayShallow(-1, -1, 0).ApplyAll(x => x*x);
        EvalResult.ExtractSubArrayShallow(-1, -1, 1).SetAll(1.0);

    },
    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
         for (int i = 0; i < Length; i++){
            errB += ResultsOfIntegration[i, 0];
            areaB += ResultsOfIntegration[i, 1];
        }
    }).Execute();
    errB = Math.Sqrt(errB);

    switch(mode){
        case 0:
        default:
            return Math.Sqrt(errA * errA + errB * errB) / b;
        case 1:
            return errB / b;
        case -1:
            return errA / b;
        case 2:
            return errB;
        case -2:
            return errA;
        case 3:
            return areaB;
        case -3:
            return areaA;
        case 4:
            return errB / Math.Sqrt(areaB);
        case -4:
            return errA / Math.Sqrt(areaA);
        case 10:
            return areaA + areaB;
        case 11:
            return Math.Sqrt(errA * errA + errB * errB);
    }
}

In [ ]:
public static Plot2Ddata L2Errors(IEnumerable<ISessionInfo> sessions){
    Plot2Ddata plot = new Plot2Ddata();
    List<double> errT = new List<double>();
    List<double> errV = new List<double>();
    List<double> h = new List<double>();

    foreach(var s in sessions){
        Console.WriteLine(s.Name);
        errT.Add(ComputeErrorTemperature(s.Timesteps.Last()));
        errV.Add(ComputeErrorVelocity(s.Timesteps.Last()));
        h.Add(Constants.L/s.Timesteps.Last().Grid.NumberOfCells);
    }

    plot.AddDataGroup("Temperature Error", h, errT, "k-x");
    plot.AddDataGroup("Velocity Error", h, errV, "k-+");
    plot.AddDataGroup(" O(1) ", new[] {h.Min(), h.Max()}, new[] {errV.Min(), h.Max()/h.Min()*errV.Min()}, "r-");

    plot.Xlabel = "cell size [m]";
    plot.Ylabel = "normalized error";
    plot.LogX = true;
    plot.LogY = true;

    return plot;
}

public static Plot2Ddata[] L2Errors(SortedDictionary<int, List<ISessionInfo>> groupedSessions, int mode = 0){
    Plot2Ddata plotT = new Plot2Ddata();
    Plot2Ddata plotV = new Plot2Ddata();

    foreach(var kvp in groupedSessions){
        List<double> errT = new List<double>();
        List<double> errV = new List<double>();
        List<double> h = new List<double>();

        foreach(var s in kvp.Value){
            var ts = s.Timesteps.Last();
            Console.WriteLine(s.Name + " t = {0}", ts.PhysicalTime);
            errT.Add(ComputeErrorTemperature(ts, mode));
            errV.Add(ComputeErrorVelocity(ts, mode));
            h.Add(Constants.L/ts.Grid.NumberOfCells);
        }

        plotT.AddDataGroup("p = " + kvp.Key, h, errT, "k-x");
        plotV.AddDataGroup("p = " + kvp.Key, h, errV, "k-x");
    }

    //plot.AddDataGroup(" O(1) ", new[] {h.Min(), h.Max()}, new[] {errV.Min(), h.Max()/h.Min()*errV.Min()}, "r-");
    plotT.ModFormat();
    plotT.Xlabel = "cell size [m]";
    plotT.Ylabel = "temperature error";
    plotT.LogX = true;
    plotT.LogY = true;

    plotV.ModFormat();
    plotV.Xlabel = "cell size [m]";
    plotV.Ylabel = "velocity error";
    plotV.LogX = true;
    plotV.LogY = true;

    return new[] {plotT, plotV};
}

#### Evaluation

In [ ]:
var plt = L2Errors(groupedSessions, -4);

In [ ]:
plt[0].PlotNow()

In [ ]:
plt[1].PlotNow()

In [ ]:
GetV(0.00221)[0](new[] {0.0, 0.0})

## Assert

In [ ]:
foreach(var kvp in refslopes){
    if(kvp.Value > slopes[kvp.Key]){
        throw new ApplicationException("Slope too low for polynomial order : " + kvp.Key);
    }
}